# Advanced Python Rich Comparisons: A New, Step-by-Step Problem Tutorial

**Topic:** `__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__`, reflected dispatch, `NotImplemented`, ordering laws, `@total_ordering`, sorting and binary search.

This notebook uses the teaching progression of the supplied **Rich Comparisons** lesson: begin with a concrete question, run a short experiment, interpret the result, extend the class, and only then arrive at a complete solution. The scenarios below are **new problems**, not reprints of the earlier 18-problem notebook.

**How to study:** Run the notebook from top to bottom. Before each experiment, predict the result. The small `assert` statements are executable checks of the stated contract; expected errors are caught and inspected rather than left as broken cells. All exercises use the Python standard library.

**Requirements:** Python 3.10+; no downloads or external packages. Each problem can be studied separately, although the shared testing helper is defined immediately below.

## One tiny test helper

An advanced comparison example often needs to demonstrate a *deliberate* `TypeError` or `ValueError`. We will test such errors explicitly instead of letting them interrupt the notebook. If an operation unexpectedly succeeds, our helper fails the test.

In [1]:
def expect_error(error_type, action):
    """Return an expected exception, or fail if it is not raised."""
    try:
        action()
    except error_type as exc:
        return exc
    raise AssertionError(f"Expected {error_type.__name__}, but no error occurred")


assert isinstance(expect_error(TypeError, lambda: 1 < object()), TypeError)
print("Testing helper ready")

Testing helper ready


---
# Problem 1 — A two-class equality handshake

**Challenge:** Two unrelated classes represent two kinds of access badge. A `Badge` knows only its serial number; a `LegacyBadge` understands how to compare itself to `Badge`. Make `Badge(17) == LegacyBadge(17)` and the reverse comparison return `True` without giving `Badge` any knowledge of the legacy class.

There is an important distinction to investigate: **declining a comparison** is not the same as **deciding it is false**.

**Prediction:** Which method should get a chance to run if `Badge.__eq__` does not recognize the other type?

### Step 1 — Build an equality method that does not guess

Return the special singleton `NotImplemented` for unsupported types. It is not an exception, and it is not a Boolean result. Python's comparison machinery interprets it as a request to try the other operand's comparison method.

In [2]:
equality_trace = []


class Badge:
    def __init__(self, serial):
        self.serial = serial

    def __eq__(self, other):
        equality_trace.append("Badge.__eq__")
        if isinstance(other, Badge):
            return self.serial == other.serial
        return NotImplemented


class LegacyBadge:
    def __init__(self, old_serial):
        self.old_serial = old_serial

    def __eq__(self, other):
        equality_trace.append("LegacyBadge.__eq__")
        if isinstance(other, Badge):
            return self.old_serial == other.serial
        if isinstance(other, LegacyBadge):
            return self.old_serial == other.old_serial
        return NotImplemented


modern, old = Badge(17), LegacyBadge(17)
result = modern == old
print("result:", result)
print("method calls:", equality_trace)
assert result is True
assert equality_trace == ["Badge.__eq__", "LegacyBadge.__eq__"]

result: True
method calls: ['Badge.__eq__', 'LegacyBadge.__eq__']


### Step 2 — Reverse the operands

When the `LegacyBadge` is on the left, its method handles the comparison immediately. We therefore expect a *different trace* but the same answer. Equality should be symmetric even though method dispatch is not necessarily symmetric.

In [3]:
equality_trace.clear()
reverse_result = old == modern
print("reverse result:", reverse_result)
print("method calls:", equality_trace)
assert reverse_result is True
assert equality_trace == ["LegacyBadge.__eq__"]

reverse result: True
method calls: ['LegacyBadge.__eq__']


### Step 3 — Examine the method directly

An explicit `modern.__eq__(object())` is a method call, not the full comparison protocol. It may return `NotImplemented`. In contrast, `modern == object()` lets Python finish its fallback sequence. If both equality methods decline, equality falls back to identity-based behavior rather than raising an ordering `TypeError`.

Do **not** use `bool(NotImplemented)` to test it; check identity with `is NotImplemented`.

In [4]:
stranger = object()
raw = modern.__eq__(stranger)
print("direct method declined:", raw is NotImplemented)
print("full == operator:", modern == stranger)
print("full != operator:", modern != stranger)
assert raw is NotImplemented
assert (modern == stranger) is False
assert (modern != stranger) is True

direct method declined: True
full == operator: False
full != operator: True


### Complete solution: what we proved

`Badge.__eq__` has only one responsibility: compare badges it understands. `LegacyBadge.__eq__` handles the bridge. `NotImplemented` preserves extensibility; returning `False` too early prevents the right operand from contributing its interpretation. This is useful for interoperability between independently authored classes.

**Try it:** Change `Badge.__eq__` to return `False` for the legacy class and predict why the first comparison changes while the reversed one may not.

---
# Problem 2 — When `!=` contradicts `==`

**Challenge:** Review an apparently harmless class whose author implemented `__ne__` independently. Diagnose an inconsistency and produce a version for which equality and inequality always agree for supported operands.

Recall: in ordinary user-defined classes, `object.__ne__` delegates to the equality result and negates it when appropriate. That convenience disappears if we override `__ne__` incorrectly.

### Step 1 — Reproduce the bug

The following author wrote `__ne__` as a comparison of *object identity*. Predict the results for two separate objects with the same identifier.

In [5]:
class BrokenAccount:
    def __init__(self, account_id):
        self.account_id = account_id

    def __eq__(self, other):
        if isinstance(other, BrokenAccount):
            return self.account_id == other.account_id
        return NotImplemented

    def __ne__(self, other):
        return self is not other  # BUG: not the logical opposite of value equality


a, b = BrokenAccount("A-42"), BrokenAccount("A-42")
print("a == b:", a == b)
print("a != b:", a != b)
assert a == b
assert a != b  # This confirms the presence of the intentional defect.

a == b: True
a != b: True


### Step 2 — Fix the simplest possible thing

We can avoid defining `__ne__` at all. The default implementation from `object` delegates to `__eq__` and negates a supported result. This makes the class smaller and removes a redundant source of disagreement.

In [6]:
class Account:
    def __init__(self, account_id):
        self.account_id = account_id

    def __eq__(self, other):
        if isinstance(other, Account):
            return self.account_id == other.account_id
        return NotImplemented


first = Account("A-42")
second = Account("A-42")
third = Account("B-19")
assert first == second and not (first != second)
assert first != third and not (first == third)
print("fixed:", first == second, first != second, first != third)

fixed: True False True


### Step 3 — What if an explicit `__ne__` is required?

For a custom override, propagate `NotImplemented` rather than negating it. The expression `not NotImplemented` is the wrong approach: the sentinel must be handled as a *dispatch signal*, not coerced to truth.

In [7]:
class ExplicitAccount(Account):
    def __ne__(self, other):
        equal = self.__eq__(other)
        if equal is NotImplemented:
            return NotImplemented
        return not equal


left, right = ExplicitAccount("same"), ExplicitAccount("same")
assert left.__ne__(object()) is NotImplemented
assert left == right and not (left != right)
print("explicit override also preserves the contract")

explicit override also preserves the contract


**Solution takeaway:** Write `__eq__` once and normally inherit `__ne__`. If a domain genuinely needs custom inequality, verify complementarity and correctly forward `NotImplemented`. Defining equality on a mutable class also normally makes it unhashable unless a separate safe hash policy is supplied; we will revisit the design of immutable value types in later problems.

---
# Problem 3 — A subclass gets the first word in equality

**Challenge:** A signed document is a specialized document. A regular document compares by body; signed documents compare by both body and signature, and they must *not* compare equal to unsigned documents. Can we enforce this rule in **both** operand orders?

When the right operand is a proper subclass that overrides the reflected comparison, Python can give the subclass's equality method priority. Let's observe rather than assume the call order.

### Step 1 — Create the document hierarchy

The base class refuses comparison against a signed document: an unsigned document cannot certify a signature. The subclass handles only exact signed-document peers. Logging makes the dispatch order visible.

In [8]:
subclass_trace = []


class Document:
    def __init__(self, body):
        self.body = body

    def __eq__(self, other):
        subclass_trace.append("Document.__eq__")
        if type(other) is Document:
            return self.body == other.body
        return NotImplemented


class SignedDocument(Document):
    def __init__(self, body, signature):
        super().__init__(body)
        self.signature = signature

    def __eq__(self, other):
        subclass_trace.append("SignedDocument.__eq__")
        if type(other) is SignedDocument:
            return (self.body, self.signature) == (other.body, other.signature)
        if isinstance(other, Document):
            return False
        return NotImplemented


plain = Document("report")
signed = SignedDocument("report", "key-7")
assert (plain == signed) is False
print("plain == signed trace:", subclass_trace)
assert subclass_trace == ["SignedDocument.__eq__"]

plain == signed trace: ['SignedDocument.__eq__']


### Step 2 — Try the reverse direction

Here the subclass is already on the left and can immediately reject the plain document. Verify that the result remains symmetric, then test two actual signed documents.

In [9]:
subclass_trace.clear()
assert (signed == plain) is False
print("signed == plain trace:", subclass_trace)
assert subclass_trace == ["SignedDocument.__eq__"]

copy = SignedDocument("report", "key-7")
other_signature = SignedDocument("report", "key-8")
assert signed == copy
assert signed != other_signature
assert Document("report") == Document("report")
print("same signature:", signed == copy, "different signature:", signed == other_signature)

signed == plain trace: ['SignedDocument.__eq__']
same signature: True different signature: False


### Step 3 — Decide which notion of equality you mean

An alternative domain might define `SignedDocument` and `Document` as equal whenever their bodies match. That is a *different contract*. It is not safe to implement it on one side only, or to let the base class ignore fields that the subclass treats as significant. Our solution explicitly chooses **different types are not equal** and tests both orders.

**Extension:** Replace the `False` in `SignedDocument.__eq__` with `NotImplemented`. What does the completed operator do once both sides decline? Check whether the result changes for distinct objects.

---
# Problem 4 — Unicode-canonical identifiers

**Challenge:** Build a case-insensitive, hashable identifier class for a directory. `Straße` and `STRASSE` should compare equal; visually equivalent composed/decomposed Unicode should also compare equal. Ordering, hashing and equality must use the *same canonical representation*.

Our domain policy is **Unicode NFKC normalization followed by case folding**. This is an explicit application-level policy, not a universally correct rule for names, security identifiers or every language.

### Step 1 — Explore two problems with naive string comparison

Direct string equality compares code points. Lowercasing alone does not handle all case-folding equivalences, and decomposed Unicode may have a different representation from composed Unicode.

In [10]:
import unicodedata

print('"Straße".lower():', "Straße".lower())
print('"Straße".casefold():', "Straße".casefold())
print("composed equals decomposed:", "é" == "e\u0301")
assert "Straße".lower() != "STRASSE".lower()
assert "Straße".casefold() == "STRASSE".casefold()
assert "é" != "e\u0301"

"Straße".lower(): straße
"Straße".casefold(): strasse
composed equals decomposed: False


### Step 2 — Encapsulate one normalization rule

The raw spelling is preserved for display, while `_canonical` is immutable comparison data. We deliberately accept only actual strings. Returning `NotImplemented` for another kind of object avoids silently interpreting arbitrary objects as names.

In [11]:
class DirectoryKey:
    __slots__ = ("_display", "_canonical")

    def __init__(self, display):
        if not isinstance(display, str):
            raise TypeError("DirectoryKey requires a string")
        object.__setattr__(self, "_display", display)
        canonical = unicodedata.normalize("NFKC", display).casefold()
        object.__setattr__(self, "_canonical", canonical)

    def __setattr__(self, name, value):
        raise AttributeError("DirectoryKey is immutable")

    def __repr__(self):
        return f"DirectoryKey({self._display!r})"

    def __eq__(self, other):
        if isinstance(other, DirectoryKey):
            return self._canonical == other._canonical
        return NotImplemented

    def __lt__(self, other):
        if isinstance(other, DirectoryKey):
            return self._canonical < other._canonical
        return NotImplemented

    def __hash__(self):
        return hash(self._canonical)


k1, k2 = DirectoryKey("Straße"), DirectoryKey("STRASSE")
k3, k4 = DirectoryKey("é"), DirectoryKey("e\u0301")
assert k1 == k2 and hash(k1) == hash(k2)
assert k3 == k4 and hash(k3) == hash(k4)
print(k1, k2, "equal:", k1 == k2)
print(k3, k4, "equal:", k3 == k4)

DirectoryKey('Straße') DirectoryKey('STRASSE') equal: True
DirectoryKey('é') DirectoryKey('é') equal: True


### Step 3 — Use it in a set and in a sort

If two values are equal, they must have equal hashes. Equality need *not* preserve the original spelling, and two equal canonical keys should tie in ordering. The operator `<` is sufficient for `sorted`; the `<=` operator is **not** magically synthesized from equality and less-than.

In [12]:
names = [DirectoryKey("Zoe"), DirectoryKey("straße"),
         DirectoryKey("ALICE"), DirectoryKey("Strasse")]
print("sorted display spellings:", [x._display for x in sorted(names)])
print("distinct canonical names:", len(set(names)))
assert len(set(names)) == 3
assert not (k1 < k2) and not (k2 < k1)
assert DirectoryKey("alice") < DirectoryKey("zoe")
expect_error(TypeError, lambda: k1 <= k2)
expect_error(AttributeError, lambda: setattr(k1, "_canonical", "altered"))

sorted display spellings: ['ALICE', 'straße', 'Strasse', 'Zoe']
distinct canonical names: 3


AttributeError('DirectoryKey is immutable')

**Complete solution:** Canonicalize once at construction, compare the canonical value consistently, make it immutable, and hash that same value. Do not pretend Python inferred `<=` for us: implement it explicitly or choose `@total_ordering` after deciding that the relation is a genuine total order.

**Challenge extension:** How would the policy change if identifiers were case-sensitive? How would you protect against unwanted NFKC equivalences in a security-sensitive application?

---
# Problem 5 — Exact rational numbers and cross-type comparisons

**Challenge:** Implement a fraction wrapper that compares exactly with `int` and `fractions.Fraction`, in both operand orders, without converting to floating point. Equal numeric values should share a hash even when their concrete types differ.

This is a deliberately supported cross-type *numeric* equality contract; it differs from comparing a fraction to a random tuple or a formatted string.

### Step 1 — Why not convert everything to float?

Binary floating-point cannot represent every rational exactly. It is particularly unsafe to compare large adjacent integers after coercing both to `float`. Use exact integer arithmetic, conveniently supplied by `Fraction`.

In [13]:
from fractions import Fraction

huge = 2**60
print("adjacent integers distinct:", huge != huge + 1)
print("after float coercion distinct:", float(huge) != float(huge + 1))
assert huge != huge + 1
assert float(huge) == float(huge + 1)
assert Fraction(huge) < Fraction(huge + 1)

adjacent integers distinct: True
after float coercion distinct: False


### Step 2 — Define accepted operand types explicitly

`bool` is a subclass of `int`, but treating `True` as a rational input would be surprising in this API. We reject booleans at construction, and support only `Rational`, exact built-in integers, and `Fraction` for comparisons.

Because we support cross-type equality, we also need to meet the cross-type hash rule.

In [14]:
class Rational:
    __slots__ = ("_value",)

    def __init__(self, numerator, denominator=1):
        if type(numerator) is not int or type(denominator) is not int:
            raise TypeError("numerator and denominator must be integers (not bool)")
        if denominator == 0:
            raise ZeroDivisionError("denominator cannot be zero")
        object.__setattr__(self, "_value", Fraction(numerator, denominator))

    def __setattr__(self, name, value):
        raise AttributeError("Rational is immutable")

    def __repr__(self):
        return f"Rational({self._value.numerator}, {self._value.denominator})"

    @staticmethod
    def _coerce(other):
        if isinstance(other, Rational):
            return other._value
        if type(other) is int or isinstance(other, Fraction):
            return Fraction(other)
        return NotImplemented

    def __eq__(self, other):
        value = self._coerce(other)
        if value is NotImplemented:
            return NotImplemented
        return self._value == value

    def __lt__(self, other):
        value = self._coerce(other)
        if value is NotImplemented:
            return NotImplemented
        return self._value < value

    def __gt__(self, other):
        value = self._coerce(other)
        if value is NotImplemented:
            return NotImplemented
        return self._value > value

    def __hash__(self):
        return hash(self._value)


half = Rational(2, 4)
whole = Rational(8, 4)
assert half == Fraction(1, 2) and Fraction(1, 2) == half
assert whole == 2 and 2 == whole
assert hash(whole) == hash(2)
assert hash(half) == hash(Fraction(1, 2))
print(half, whole, "hash compatible:", hash(whole) == hash(2))

Rational(1, 2) Rational(2, 1) hash compatible: True


### Step 3 — Test reflected ordering and unsupported types

An `int` or `Fraction` may decline comparison with our wrapper, after which Python can try the wrapper's reflected partner. We implemented `Rational.__gt__` explicitly so `1 < Rational(3, 2)` works; `Rational.__lt__` supports the opposite reflection. The unsupported string case remains *unsupported* for ordering rather than being given arbitrary numerical meaning.

In [15]:
assert Rational(1, 3) < Rational(1, 2)
assert 1 < Rational(3, 2)
assert Rational(3, 2) > 1
assert Fraction(3, 2) > Rational(1, 2)
assert Rational(huge) < Rational(huge + 1)
assert Rational(1, 2) != "half"
expect_error(TypeError, lambda: Rational(1, 2) < "half")
expect_error(TypeError, lambda: Rational(True, 1))
expect_error(ZeroDivisionError, lambda: Rational(1, 0))
print("exact, reflected, and unsupported-type checks passed")

exact, reflected, and unsupported-type checks passed


### Step 4 — Find a subtle missing method

Our `Rational` provides `__eq__`, `__lt__`, and `__gt__` for cross-type reflection. But `<=` still needs its own method (or a suitable reflected `>=`) and is not inferred automatically.

In [16]:
expect_error(TypeError, lambda: Rational(1, 2) <= Rational(2, 3))
print("<= still needs explicit support")

<= still needs explicit support


**Solution takeaway:** Cross-type comparisons are possible when they have a precise mathematical meaning. Coerce only recognized types, avoid lossy conversions, preserve `NotImplemented`, and keep hash semantics consistent with numeric equality. Later, we will let `@total_ordering` generate the other ordering operators for a different, genuinely total-order type.

---
# Problem 6 — Compare instants, not clock-face strings

**Challenge:** Two time-zone-aware datetimes can display different clock times while describing the *same instant*. Implement a wrapper that compares instants in UTC, rejects naive datetimes and stays immutable.

This problem is about a comparison policy, not about formatting timestamps or inferring a missing time zone.

### Step 1 — Set up equal instants with different offsets

The local clock values differ, but they refer to the same point on the timeline. Notice that string comparison is not a time-zone conversion algorithm.

In [17]:
from datetime import datetime, timedelta, timezone

utc_time = datetime(2025, 3, 7, 10, 0, tzinfo=timezone.utc)
plus_two = datetime(2025, 3, 7, 12, 0,
                    tzinfo=timezone(timedelta(hours=2)))
print("same instant:", utc_time == plus_two)
print("different text:", utc_time.isoformat() != plus_two.isoformat())
assert utc_time == plus_two
assert utc_time.isoformat() != plus_two.isoformat()

same instant: True
different text: True


### Step 2 — Normalize the value once

A datetime is considered aware for our purposes when its `utcoffset()` is not `None`. Reject naive values at the boundary rather than silently assuming UTC or the machine's local time zone. We compare only instances of `Instant`.

In [18]:
class Instant:
    __slots__ = ("_utc",)

    def __init__(self, value):
        if not isinstance(value, datetime) or value.utcoffset() is None:
            raise ValueError("Instant requires a time-zone-aware datetime")
        object.__setattr__(self, "_utc", value.astimezone(timezone.utc))

    def __setattr__(self, name, value):
        raise AttributeError("Instant is immutable")

    def __repr__(self):
        return f"Instant({self._utc.isoformat()})"

    def __eq__(self, other):
        if isinstance(other, Instant):
            return self._utc == other._utc
        return NotImplemented

    def __lt__(self, other):
        if isinstance(other, Instant):
            return self._utc < other._utc
        return NotImplemented

    def __hash__(self):
        return hash(self._utc)


x = Instant(utc_time)
y = Instant(plus_two)
later = Instant(datetime(2025, 3, 7, 10, 1, tzinfo=timezone.utc))
assert x == y and hash(x) == hash(y)
assert x < later and y < later
print(x, y, "equal:", x == y)

Instant(2025-03-07T10:00:00+00:00) Instant(2025-03-07T10:00:00+00:00) equal: True


### Step 3 — Prove that invalid inputs fail early

Comparison errors should not depend on what time zone happens to be configured on the host computer. Explicit validation moves ambiguity to construction, where the caller has enough context to supply the intended zone.

In [19]:
naive = datetime(2025, 3, 7, 10, 0)
expect_error(ValueError, lambda: Instant(naive))
expect_error(ValueError, lambda: Instant("2025-03-07T10:00Z"))
expect_error(TypeError, lambda: x < utc_time)
expect_error(AttributeError, lambda: setattr(x, "_utc", later._utc))
assert len({x, y, later}) == 2
print("invalid-input, immutability and hashing tests passed")

invalid-input, immutability and hashing tests passed


**Complete solution:** Normalize to aware UTC at construction and compare only normalized values. The instance's displayed offset is not the definition of equality. The same boundary-validation technique is useful when implementing value objects for measurements, identifiers and money.

---
# Problem 7 — Pareto dominance is a partial order

**Challenge:** Compare two computing configurations on both **cost** and **latency**, where lower is better. Configuration A dominates B if A is no worse on either dimension and strictly better on at least one. Two trade-offs may be *incomparable*.

Your task is to implement honest rich comparisons **without pretending that every pair can be ranked**.

### Step 1 — Write down the mathematical relation

For points `a = (cost_a, latency_a)` and `b = (cost_b, latency_b)`:

- `a <= b`: both coordinates of `a` are less than or equal to the corresponding coordinates of `b`;
- `a < b`: `a <= b` **and** at least one coordinate is strictly smaller;
- `a == b`: both coordinates are equal.

If one configuration costs less but runs more slowly, neither `<` nor `>` holds. This is *not* the same thing as equality.

In [20]:
from dataclasses import dataclass


@dataclass(frozen=True, slots=True)
class Configuration:
    cost: int
    latency: int

    def __le__(self, other):
        if not isinstance(other, Configuration):
            return NotImplemented
        return self.cost <= other.cost and self.latency <= other.latency

    def __lt__(self, other):
        if not isinstance(other, Configuration):
            return NotImplemented
        return self <= other and self != other

    def __ge__(self, other):
        if not isinstance(other, Configuration):
            return NotImplemented
        return other <= self

    def __gt__(self, other):
        if not isinstance(other, Configuration):
            return NotImplemented
        return other < self


cheap_slow = Configuration(40, 100)
expensive_fast = Configuration(90, 20)
strictly_better = Configuration(30, 80)
print("cheap_slow < expensive_fast:", cheap_slow < expensive_fast)
print("expensive_fast < cheap_slow:", expensive_fast < cheap_slow)
print("strictly_better < cheap_slow:", strictly_better < cheap_slow)
assert not (cheap_slow < expensive_fast)
assert not (expensive_fast < cheap_slow)
assert cheap_slow != expensive_fast
assert strictly_better < cheap_slow

cheap_slow < expensive_fast: False
expensive_fast < cheap_slow: False
strictly_better < cheap_slow: True


### Step 2 — Expose all four directions of incomparability

With a partial order, it is possible that **every one** of `<`, `>`, `<=` and `>=` returns `False` for two distinct valid values. An application should not automatically turn that case into an arbitrary winner.

In [21]:
comparisons = {
    "<": cheap_slow < expensive_fast,
    ">": cheap_slow > expensive_fast,
    "<=": cheap_slow <= expensive_fast,
    ">=": cheap_slow >= expensive_fast,
}
print("trade-off comparison:", comparisons)
assert comparisons == {"<": False, ">": False, "<=": False, ">=": False}
assert cheap_slow <= cheap_slow
assert not (cheap_slow < cheap_slow)

trade-off comparison: {'<': False, '>': False, '<=': False, '>=': False}


### Step 3 — Why blindly using `@total_ordering` would mislead

`@total_ordering` generates missing methods using formulas appropriate to a total order. For example, a generated `>=` can act as `not (a < b)`. For incomparable configurations, that would incorrectly report that one is at least as good as the other.

We demonstrate the *bad* idea in a separate class so it does not contaminate our correct solution.

In [22]:
from functools import total_ordering


@total_ordering
class BrokenDominance:
    def __init__(self, cost, latency):
        self.cost, self.latency = cost, latency

    def __eq__(self, other):
        if not isinstance(other, BrokenDominance):
            return NotImplemented
        return (self.cost, self.latency) == (other.cost, other.latency)

    def __lt__(self, other):
        if not isinstance(other, BrokenDominance):
            return NotImplemented
        return (self.cost <= other.cost and
                self.latency <= other.latency and self != other)


bad_a, bad_b = BrokenDominance(40, 100), BrokenDominance(90, 20)
print("a < b:", bad_a < bad_b)
print("generated a >= b:", bad_a >= bad_b)
assert not (bad_a < bad_b)
assert bad_a >= bad_b  # Incorrect for Pareto dominance: intentional counterexample.

a < b: False
generated a >= b: True


### Step 4 — Solve the sorting requirement separately

A product interface may still need a linear *display order*. Choose an explicit display policy, for example `(cost, latency)`, and pass it to `sorted(key=...)`. This does **not** redefine Pareto dominance: it is a different, total-order presentation key.

In [23]:
options = [expensive_fast, cheap_slow, strictly_better]
by_display_policy = sorted(options, key=lambda item: (item.cost, item.latency))
print("cost-first display:", by_display_policy)
assert by_display_policy == [strictly_better, cheap_slow, expensive_fast]
assert cheap_slow != expensive_fast

cost-first display: [Configuration(cost=30, latency=80), Configuration(cost=40, latency=100), Configuration(cost=90, latency=20)]


**Complete solution:** Implement each of the four order methods when the domain relation is partial. Reserve `@total_ordering` for genuine total orders. An incomparable result is meaningful information, not an implementation failure. If a UI needs a deterministic list, give that UI its own explicit sort key.

---
# Problem 8 — Rock, paper, scissors and a non-transitive comparator

**Challenge:** A teammate argues that a sorter can use the rule "A is less than B whenever B beats A." This rule compares every pair of distinct moves, but does it define a valid sort order?

The crucial ordering law is **transitivity**: if `a < b` and `b < c`, then `a < c`. A game with a cycle intentionally violates it.

### Step 1 — Implement the tempting comparator

We define the cycle `rock < paper < scissors < rock`. Note that the final link closes the cycle rather than continuing a linear ranking.

In [24]:
class GameMove:
    _moves = frozenset({"rock", "paper", "scissors"})
    _loses_to = {("rock", "paper"),
                 ("paper", "scissors"),
                 ("scissors", "rock")}

    def __init__(self, name):
        if name not in self._moves:
            raise ValueError("unknown move")
        self.name = name

    def __repr__(self):
        return f"GameMove({self.name!r})"

    def __eq__(self, other):
        if not isinstance(other, GameMove):
            return NotImplemented
        return self.name == other.name

    def __lt__(self, other):
        if not isinstance(other, GameMove):
            return NotImplemented
        return (self.name, other.name) in self._loses_to


rock, paper, scissors = (GameMove(name)
                         for name in ("rock", "paper", "scissors"))
assert rock < paper
assert paper < scissors
assert scissors < rock
print("rock < paper:", rock < paper)
print("paper < scissors:", paper < scissors)
print("scissors < rock:", scissors < rock)

rock < paper: True
paper < scissors: True
scissors < rock: True


### Step 2 — Exhibit a transitivity counterexample

From the first two comparisons, transitivity would require `rock < scissors`. But the game specifies the opposite. This is a *semantic* defect in a proposed sorting relation, not an issue that a better sorting algorithm can fix.

In [25]:
assert rock < paper and paper < scissors
assert not (rock < scissors)
print("transitivity counterexample: rock < paper < scissors, but rock < scissors is False")

transitivity counterexample: rock < paper < scissors, but rock < scissors is False


### Step 3 — Can `sorted()` repair the relation?

No. A sorting algorithm assumes its comparison relation behaves consistently. It may still produce some list, but the game gives no linear ordering that satisfies all three edges. Avoid making tests that depend on one particular permutation from a broken comparator.

In [26]:
ordered_by_game = sorted([rock, paper, scissors])
print("one possible sort result:", ordered_by_game)
print("satisfies all game edges:",
      all(ordered_by_game.index(left) < ordered_by_game.index(right)
          for left, right in ((rock, paper), (paper, scissors), (scissors, rock))))
assert not all(ordered_by_game.index(left) < ordered_by_game.index(right)
               for left, right in ((rock, paper), (paper, scissors), (scissors, rock)))

one possible sort result: [GameMove('rock'), GameMove('paper'), GameMove('scissors')]
satisfies all game edges: False


### Step 4 — Separate a game result from a display order

If the application needs a list, define a separate display sequence. Here it is alphabetical. The game result continues to use `GameMove.__lt__`, while a UI list uses `key=lambda move: move.name`.

In [27]:
menu = sorted([rock, paper, scissors], key=lambda move: move.name)
print("display order:", [move.name for move in menu])
assert [move.name for move in menu] == ["paper", "rock", "scissors"]
assert scissors < rock  # Game behavior has not changed.

display order: ['paper', 'rock', 'scissors']


**Complete solution:** No coherent total order can implement a strict rock–paper–scissors cycle. Use a dedicated game-result relation to describe who beats whom, and an explicit, independent sorting key for display. Testing ordering *laws* can reveal defects that test cases for individual operator calls miss.

---
# Problem 9 — Sort hierarchical labels with a documented segment policy

**Challenge:** Paths such as `('build', 2)`, `('build', 10)` and `('build', 2, 'debug')` have a natural hierarchical interpretation. We want integer segments ordered numerically, textual segments ordered by normalized casefolded spelling, and prefixes before their extensions. Define a policy for mixed segment types rather than relying on accidental Python comparisons.

This is a **total order** after normalization, so here `@total_ordering` is appropriate.

### Step 1 — See the two naive failures

Stringifying integers sorts `'10'` before `'2'` lexicographically. Directly comparing `('build', 2)` with `('build', 'two')` raises a `TypeError` when it reaches `2 < 'two'`. Neither behavior defines the requested ordering policy.

In [28]:
print("string ordering:", sorted(["2", "10"]))
assert sorted(["2", "10"]) == ["10", "2"]
expect_error(TypeError, lambda: ("build", 2) < ("build", "two"))

string ordering: ['10', '2']


TypeError("'<' not supported between instances of 'int' and 'str'")

### Step 2 — Construct a comparable key for every segment

We tag each segment: `(0, integer)` for numbers and `(1, canonical_text)` for text. That choice means *numbers sort before text* when the two segment types share a position. The policy is explicit and can be changed without touching Python's global comparison behavior.

A canonical tuple of tagged segments supports tuple's built-in lexicographic comparisons, including the short-prefix rule.

In [29]:
def canonical_segment(part):
    if type(part) is int:
        return (0, part)
    if isinstance(part, str):
        return (1, unicodedata.normalize("NFKC", part).casefold())
    raise TypeError("segments must be int (not bool) or str")


assert canonical_segment(2) < canonical_segment("2")
assert canonical_segment("Straße") == canonical_segment("STRASSE")
expect_error(TypeError, lambda: canonical_segment(True))
print("numeric segment:", canonical_segment(12))
print("text segment:", canonical_segment("Build"))

numeric segment: (0, 12)
text segment: (1, 'build')


### Step 3 — Implement just `__eq__` and `__lt__`

`@total_ordering` derives `<=`, `>`, and `>=` from these two methods. The decorator does not infer that the *domain* has a total order; that is our job, established by the tagged tuple key. All six operators then use the same canonical representation.

In [30]:
@total_ordering
class PathLabel:
    __slots__ = ("_parts", "_key")

    def __init__(self, parts):
        raw = tuple(parts)
        canonical = tuple(canonical_segment(part) for part in raw)
        object.__setattr__(self, "_parts", raw)
        object.__setattr__(self, "_key", canonical)

    def __setattr__(self, name, value):
        raise AttributeError("PathLabel is immutable")

    def __repr__(self):
        return f"PathLabel({self._parts!r})"

    def __eq__(self, other):
        if isinstance(other, PathLabel):
            return self._key == other._key
        return NotImplemented

    def __lt__(self, other):
        if isinstance(other, PathLabel):
            return self._key < other._key
        return NotImplemented

    def __hash__(self):
        return hash(self._key)


p2 = PathLabel(("build", 2))
p10 = PathLabel(("build", 10))
p2_debug = PathLabel(("build", 2, "debug"))
assert p2 < p10
assert p2 < p2_debug
assert p2 <= p2_debug
assert p10 > p2
assert p10 >= p2
assert p2 == PathLabel(("BUILD", 2))
print("numeric order and prefix rule:", sorted([p10, p2_debug, p2]))

numeric order and prefix rule: [PathLabel(('build', 2)), PathLabel(('build', 2, 'debug')), PathLabel(('build', 10))]


### Step 4 — Test cross-type boundaries and hashing

Sorting is meaningful for `PathLabel` objects because every accepted segment has a total key. That does not imply a `PathLabel` should compare directly with a list, tuple, float or random object. Return `NotImplemented` for unsupported operands.

In [31]:
assert PathLabel(("STRASSE", 3)) == PathLabel(("straße", 3))
assert hash(PathLabel(("STRASSE", 3))) == hash(PathLabel(("straße", 3)))
assert p2.__lt__(("build", 2)) is NotImplemented
assert p2 != ("build", 2)
expect_error(TypeError, lambda: p2 < ("build", 2))
expect_error(TypeError, lambda: PathLabel(("build", False)))
expect_error(AttributeError, lambda: setattr(p2, "_key", ()))
assert len({p2, PathLabel(("BUILD", 2)), p10}) == 2
print("complete and consistent comparison/hash contract passed")

complete and consistent comparison/hash contract passed


**Complete solution:** Convert each accepted input segment to one comparable, canonical representation. Define equality and `<` in terms of that same representation; only then use `@total_ordering`. The tuple key already handles hierarchy, prefixes, mixed types and numeric ordering coherently.

**Extension:** What if text and numbers must be forbidden at the same depth? Change construction validation, not the comparison method.

---
# Problem 10 — Binary search is a comparison-contract test

**Challenge:** Maintain a sorted sequence of event records with numeric priority. Repeated priorities are allowed, and payload text must not affect ordering. Locate the left and right boundaries of a priority with `bisect_left` and `bisect_right`.

Unlike an equality lookup, binary search uses the sequence's ordering contract. We will use records as both stored elements and search probes, so all comparisons are well-defined.

### Step 1 — Make two records with the same sorting identity

The payload is descriptive data. Equality and less-than intentionally consider only priority. This is an explicit domain choice, so don't use such records as keys when the payload is intended to distinguish identity.

In [32]:
from dataclasses import field


@dataclass(frozen=True, slots=True)
class PriorityRecord:
    priority: int
    payload: str = field(compare=False)

    def __lt__(self, other):
        if isinstance(other, PriorityRecord):
            return self.priority < other.priority
        return NotImplemented


one = PriorityRecord(4, "alpha")
two = PriorityRecord(4, "beta")
assert one == two
assert not (one < two) and not (two < one)
assert hash(one) == hash(two)
print("same comparison value, different display payload:", one, two)

same comparison value, different display payload: PriorityRecord(priority=4, payload='alpha') PriorityRecord(priority=4, payload='beta')


### Step 2 — Prepare and validate a sorted collection

`bisect` assumes the input is already ordered. It does not sort or validate the collection. For the sequence below, duplicate priorities appear consecutively because `<` examines only the priority.

In [33]:
import bisect

records = [PriorityRecord(p, name) for p, name in
           [(1, "a"), (4, "b"), (4, "c"), (4, "d"), (9, "e")]]
assert all(not records[i + 1] < records[i]
           for i in range(len(records) - 1))
print("priorities:", [record.priority for record in records])

priorities: [1, 4, 4, 4, 9]


### Step 3 — Find both boundaries, then insert safely

`bisect_left` computes a position before existing equivalent entries; `bisect_right` computes a position after them. No payload comparison is necessary. Insert a new record at the right boundary if you want to place it after existing equal-priority records.

In [34]:
probe = PriorityRecord(4, "SEARCH PROBE")
left = bisect.bisect_left(records, probe)
right = bisect.bisect_right(records, probe)
print("matching slice indices:", left, right)
print("matching payloads:", [r.payload for r in records[left:right]])
assert (left, right) == (1, 4)
assert [r.payload for r in records[left:right]] == ["b", "c", "d"]

updated = records.copy()
updated.insert(right, PriorityRecord(4, "new"))
assert [r.payload for r in updated[1:5]] == ["b", "c", "d", "new"]
assert updated == sorted(updated)

matching slice indices: 1 4
matching payloads: ['b', 'c', 'd']


### Step 4 — Compare two interface choices

If you only need to search by a numeric key, an array of numeric keys can be simpler than an ordering-aware class. Recent Python versions also offer a `key` argument to `bisect`, but its application to search values differs from its application to array elements. A parallel key list avoids that subtlety here.

In [35]:
priorities = [record.priority for record in records]
assert bisect.bisect_left(priorities, 4) == left
assert bisect.bisect_right(priorities, 4) == right
expect_error(TypeError, lambda: bisect.bisect_left(records, 4))
print("numeric-key and object-probe approaches agree")

numeric-key and object-probe approaches agree


**Complete solution:** Decide what makes two records equivalent *for ordering*, store the sequence in that order, and search with a compatible probe. `bisect_left` and `bisect_right` are not substitutes for a correct `__lt__`, and `bisect` never repairs an unsorted input list.

---
# Problem 11 — Which operator do `min()` and `max()` actually need?

**Challenge:** A custom object implements `__lt__` only. Determine whether both `min` and `max` can still work, and use a trace to explain why. Then test the common but incorrect assumption that `<=` must also work.

Python's rich-comparison reflection can help with `<` and `>`, but it does not synthesize arbitrary compound comparisons.

### Step 1 — Create a minimal ordering probe

Record each `__lt__` call to reveal which operand was used as `self`. Our type offers no explicit `__gt__`, `__le__`, or `__ge__`.

In [36]:
minmax_trace = []


class RankOnly:
    def __init__(self, rank):
        self.rank = rank

    def __repr__(self):
        return f"RankOnly({self.rank})"

    def __lt__(self, other):
        minmax_trace.append((self.rank, "<", getattr(other, "rank", None)))
        if isinstance(other, RankOnly):
            return self.rank < other.rank
        return NotImplemented


items = [RankOnly(8), RankOnly(3), RankOnly(5)]
minimum = min(items)
print("minimum:", minimum)
print("trace:", minmax_trace)
assert minimum.rank == 3
assert minmax_trace

minimum: RankOnly(3)
trace: [(3, '<', 8), (5, '<', 3)]


### Step 2 — Test `max()` and inspect reflection

`max` uses `>` internally. For a pair of our objects, a missing `__gt__` can be reflected into the right object's `__lt__` method. That is why max can work even when no `__gt__` appears in the class definition.

In [37]:
minmax_trace.clear()
maximum = max(items)
print("maximum:", maximum)
print("reflected < calls:", minmax_trace)
assert maximum.rank == 8
assert minmax_trace
assert items[0] > items[1]

maximum: RankOnly(8)
reflected < calls: [(8, '<', 3), (8, '<', 5)]


### Step 3 — Confirm what is *not* inferred

Python reflects `a <= b` as `b >= a`, but neither method exists in this class. Python does not combine `__eq__` with `__lt__` on your behalf.

In [38]:
expect_error(TypeError, lambda: items[1] <= items[0])
expect_error(TypeError, lambda: items[0] >= items[1])
print("<= and >= are not automatically generated")

<= and >= are not automatically generated


### Step 4 — Avoid overloading operators if the order is local

For an object that has several possible rankings (e.g., rank, timestamp, cost), there may be no single obvious `<` for the class. The functions `min`, `max` and `sorted` all accept `key=`, allowing the caller to state the desired policy explicitly.

In [39]:
plain_rows = [{"rank": 8, "label": "x"},
              {"rank": 3, "label": "y"},
              {"rank": 5, "label": "z"}]
assert min(plain_rows, key=lambda row: row["rank"])["label"] == "y"
assert max(plain_rows, key=lambda row: row["rank"])["label"] == "x"
print("key= avoids creating a global ordering on dictionaries")

key= avoids creating a global ordering on dictionaries


**Complete solution:** Method reflection explains why `<` alone can support both `min` and `max` for cooperating same-type values. It does **not** imply Python has built every comparison operator. Prefer `key=` when the ordering is a context-specific query rather than an intrinsic property of the type.

---
# Problem 12 — A mutation silently breaks a sorted index

**Challenge:** You have an initially sorted list of records. A caller mutates an ordering field after the list is sorted. Nothing raises an exception, but binary search can now return unreliable answers. Demonstrate the issue, then design a safer immutable workflow.

This is different from the usual hashability discussion: even unhashable mutable objects can corrupt a *sorted-list invariant*.

### Step 1 — Sort records that have mutable ranking fields

This first design intentionally allows changing `rank` at any time. The list begins in correct order.

In [40]:
class MutableRank:
    def __init__(self, rank, name):
        self.rank, self.name = rank, name

    def __repr__(self):
        return f"MutableRank({self.rank}, {self.name!r})"

    def __lt__(self, other):
        if isinstance(other, MutableRank):
            return self.rank < other.rank
        return NotImplemented


mutable = [MutableRank(rank, f"r{rank}") for rank in (1, 5, 9)]
assert all(not mutable[i + 1] < mutable[i] for i in range(len(mutable) - 1))
print("before mutation:", mutable)

before mutation: [MutableRank(1, 'r1'), MutableRank(5, 'r5'), MutableRank(9, 'r9')]


### Step 2 — Mutate one record without re-sorting

After assigning a new rank, the *same list* contains `[1, 100, 9]`. Its sorted invariant is false. A call to `bisect` has no obligation to find a correct location in this malformed input; we test the broken invariant rather than depending on incidental search output.

In [41]:
mutable[1].rank = 100
print("after mutation:", mutable)
is_sorted = all(not mutable[i + 1] < mutable[i]
                for i in range(len(mutable) - 1))
print("still sorted:", is_sorted)
assert not is_sorted

after mutation: [MutableRank(1, 'r1'), MutableRank(100, 'r5'), MutableRank(9, 'r9')]
still sorted: False


### Step 3 — Replace objects rather than changing their comparison key

A frozen dataclass makes the failure immediate. Its ordering is generated from `rank` only; `name` is excluded by an explicit field policy. `dataclasses.replace` produces a new instance without mutating the old one.

In [42]:
from dataclasses import replace


@dataclass(frozen=True, order=True, slots=True)
class FrozenRank:
    rank: int
    name: str = field(compare=False)


frozen = [FrozenRank(rank, f"r{rank}") for rank in (1, 5, 9)]
expect_error(Exception, lambda: setattr(frozen[1], "rank", 100))
changed = replace(frozen[1], rank=100)
assert frozen[1].rank == 5 and changed.rank == 100
rebuilt = sorted([frozen[0], changed, frozen[2]])
print("rebuilt:", rebuilt)
assert [entry.rank for entry in rebuilt] == [1, 9, 100]

rebuilt: [FrozenRank(rank=1, name='r1'), FrozenRank(rank=9, name='r9'), FrozenRank(rank=100, name='r5')]


### Step 4 — Validate after every update

Immutability prevents *in-place key drift*, but it does not automatically maintain a sorted collection: you still need to insert the replacement at its new position or rebuild the list. An invariant check documents what downstream binary search is allowed to assume.

In [43]:
def sorted_by_rich_comparison(values):
    return all(not values[i + 1] < values[i]
               for i in range(len(values) - 1))


assert not sorted_by_rich_comparison(mutable)
assert sorted_by_rich_comparison(frozen)
assert sorted_by_rich_comparison(rebuilt)
assert bisect.bisect_left(rebuilt, FrozenRank(9, "probe")) == 1
print("immutable rebuild and bisect invariants passed")

immutable rebuild and bisect invariants passed


**Complete solution:** A valid rich comparison implementation is not enough: the *collection* must preserve its sorted invariant over time. Prefer immutable sort keys, make changes through replacement, and restore order before performing binary search or assuming adjacent elements are ordered.

---
# Problem 13 — When `==` returns a non-Boolean object

**Challenge:** A developer returns componentwise comparison results from a vector's `__eq__`. Python allows rich-comparison methods to return non-Boolean objects. Diagnose why that can be surprising in `if`, `assert`, container membership and general-purpose scalar APIs.

This example uses only the standard library and intentionally avoids the external behavior of any specific array package.

### Step 1 — Implement the tempting componentwise equality

The following equality returns a **tuple of Boolean values**, not a Boolean. The expression `a == b` can therefore return a tuple directly. The tuple is nonempty even when some components are `False`.

In [44]:
class ComponentMask:
    def __init__(self, values):
        self.values = tuple(values)

    def __eq__(self, other):
        if not isinstance(other, ComponentMask):
            return NotImplemented
        if len(self.values) != len(other.values):
            return False
        return tuple(a == b for a, b in zip(self.values, other.values))


mask = ComponentMask((1, 2)) == ComponentMask((1, 99))
print("raw equality result:", mask)
print("truth of result in an if:", bool(mask))
assert mask == (True, False)
assert bool(mask) is True  # BUG for a scalar "all components equal" interpretation.

raw equality result: (True, False)
truth of result in an if: True


### Step 2 — Write down the intended scalar contract

For a scalar value type, `==` should answer **one yes/no question**. Componentwise comparison is a separate operation with a separate method name. We also need to check shape before using `zip`, because `zip` stops at the shorter input.

In [45]:
class SafeVector:
    def __init__(self, values):
        self.values = tuple(values)

    def equal_components(self, other):
        if not isinstance(other, SafeVector):
            raise TypeError("expected SafeVector")
        if len(self.values) != len(other.values):
            raise ValueError("vectors must have matching dimensions")
        return tuple(a == b for a, b in zip(self.values, other.values))

    def __eq__(self, other):
        if not isinstance(other, SafeVector):
            return NotImplemented
        return self.values == other.values


v_a, v_b = SafeVector((1, 2)), SafeVector((1, 99))
assert (v_a == v_b) is False
assert v_a.equal_components(v_b) == (True, False)
assert v_a == SafeVector((1, 2))
expect_error(ValueError, lambda: v_a.equal_components(SafeVector((1,))))
print("scalar equality:", v_a == v_b)
print("explicit componentwise result:", v_a.equal_components(v_b))

scalar equality: False
explicit componentwise result: (True, False)


### Step 3 — Clarify the general rule

The Python data model *permits* rich-comparison methods to produce a non-Boolean value. Some vectorized APIs intentionally do so and define special rules about truth testing. For an ordinary scalar domain object, however, Boolean equality is clearer and composes predictably with collections and branching. If your class intentionally returns a mask, document its truth-value behavior explicitly.

**Complete solution:** Make `SafeVector.__eq__` return one Boolean and expose the componentwise operation under a separate name. Remember: `NotImplemented` signals unsupported operand dispatch; it is not a stand-in for a componentwise mask.

---
# Problem 14 — Capstone: an exactly comparable unit-aware length

**Challenge:** Design an immutable `Length` object that accepts millimeters (`mm`), centimeters (`cm`) and meters (`m`). It must:

1. Convert inputs to an *exact* canonical unit.
2. Treat `Length('0.1', 'cm')` and `Length(1, 'mm')` as equal and give them equal hashes.
3. Support all six rich-comparison operators with a genuine total order.
4. Reject unsupported units, non-finite decimals, booleans and floating-point inputs.
5. Work correctly with `sorted`, `bisect` and sets.

Rather than writing one enormous class first, we will solve and test the representation, equality, ordering and integration in separate steps.

### Step 1 — Choose the representation before operators

Represent the canonical length in millimeters as a `Fraction`. `Decimal('0.1')` converts to the exact fraction `1/10`; after multiplying by 10 millimeters per centimeter, the result is exactly one millimeter. Native float input is rejected instead of silently importing a binary approximation.

A unit scale maps `mm → 1`, `cm → 10` and `m → 1000` millimeters.

In [46]:
from decimal import Decimal, InvalidOperation

UNIT_TO_MM = {"mm": Fraction(1),
              "cm": Fraction(10),
              "m": Fraction(1000)}


def exact_number(value):
    """Accept only explicit exact numeric representations."""
    if type(value) is int or isinstance(value, Fraction):
        return Fraction(value)
    if isinstance(value, Decimal):
        if not value.is_finite():
            raise ValueError("amount must be finite")
        return Fraction(value)
    if isinstance(value, str):
        try:
            amount = Decimal(value)
        except InvalidOperation as exc:
            raise ValueError("invalid decimal string") from exc
        if not amount.is_finite():
            raise ValueError("amount must be finite")
        return Fraction(amount)
    raise TypeError("amount must be int, Fraction, Decimal, or decimal string")


assert exact_number("0.1") * UNIT_TO_MM["cm"] == Fraction(1)
assert exact_number(Decimal("0.25")) == Fraction(1, 4)
expect_error(TypeError, lambda: exact_number(0.1))
expect_error(TypeError, lambda: exact_number(True))
expect_error(ValueError, lambda: exact_number("NaN"))
print("canonical-unit representation checks passed")

canonical-unit representation checks passed


### Step 2 — Construct a frozen value object

Store the original input for display, plus a computed private `_mm` field. A frozen dataclass keeps both values stable after construction. We use `eq=False` because generated field-by-field equality would wrongly distinguish the different units and spellings of the *same physical length*.

In [47]:
@dataclass(frozen=True, slots=True, eq=False)
class Length:
    amount: str | int | Decimal | Fraction
    unit: str
    _mm: Fraction = field(init=False, repr=False)

    def __post_init__(self):
        if self.unit not in UNIT_TO_MM:
            raise ValueError(f"unsupported unit: {self.unit!r}")
        mm = exact_number(self.amount) * UNIT_TO_MM[self.unit]
        object.__setattr__(self, "_mm", mm)

    def __eq__(self, other):
        if isinstance(other, Length):
            return self._mm == other._mm
        return NotImplemented

    def __lt__(self, other):
        if isinstance(other, Length):
            return self._mm < other._mm
        return NotImplemented

    def __hash__(self):
        return hash(self._mm)


one_mm = Length(1, "mm")
one_mm_alternative = Length("0.1", "cm")
assert one_mm == one_mm_alternative
assert hash(one_mm) == hash(one_mm_alternative)
assert one_mm._mm == Fraction(1)
print(one_mm, "==", one_mm_alternative, ":", one_mm == one_mm_alternative)

Length(amount=1, unit='mm') == Length(amount='0.1', unit='cm') : True


### Step 3 — Add remaining ordering operations deliberately

For this domain, every accepted `Length` is represented by a finite exact rational in one dimension, so a total order exists. `@total_ordering` is appropriate. It is applied *after* the class definition to make the incremental build visible, just as a tutorial can evolve an initial class one method at a time.

The decorator fills missing ordering operations and preserves our existing `__eq__` and `__lt__`.

In [48]:
Length = total_ordering(Length)

small = Length("0.1", "mm")
medium = Length(1, "mm")
large = Length("0.002", "m")
assert small < medium < large
assert medium <= one_mm_alternative
assert medium >= one_mm_alternative
assert large > medium
assert large != medium
print("all six operators work:", small < medium, medium <= one_mm_alternative,
      large > medium, large >= small, large != medium, medium == one_mm_alternative)

all six operators work: True True True True True True


### Step 4 — Verify reflected comparisons and unsafe inputs

Reflection is useful for compatible `Length` operands regardless of which instance appears first. It does not justify an implicit comparison with raw numbers: `1` might mean one millimeter, one meter or a dimensionless value. Reject it rather than silently picking a unit.

In [49]:
assert Length(2, "mm") > Length(1, "mm")
assert Length(1, "mm") < Length(2, "mm")
assert Length(1, "cm") == Length(10, "mm")
assert Length(1, "mm") != "1 mm"
assert one_mm.__lt__(1) is NotImplemented
expect_error(TypeError, lambda: one_mm < 1)
expect_error(ValueError, lambda: Length(1, "yard"))
expect_error(ValueError, lambda: Length("Infinity", "mm"))
expect_error(TypeError, lambda: Length(1.0, "mm"))
expect_error(TypeError, lambda: Length(False, "mm"))
expect_error(Exception, lambda: setattr(one_mm, "unit", "m"))
print("type, unit, finiteness, immutability boundaries passed")

type, unit, finiteness, immutability boundaries passed


### Step 5 — Prove compatibility with ordinary Python collections

A sorted collection follows exact millimeter values regardless of input units. A set combines physically equal values using the canonical hash. A `bisect` search uses a `Length` probe in a compatible comparison domain.

In [50]:
lengths = [Length(2, "cm"), Length(1, "mm"),
           Length("0.005", "m"), Length(10, "mm")]
ordered_lengths = sorted(lengths)
print("sorted millimeters:", [item._mm for item in ordered_lengths])
assert [item._mm for item in ordered_lengths] == [Fraction(1), Fraction(5),
                                                  Fraction(10), Fraction(20)]
assert len({Length(1, "mm"), Length("0.1", "cm"), Length("0.001", "m")}) == 1
insertion = bisect.bisect_left(ordered_lengths, Length(1, "cm"))
assert insertion == 2
assert ordered_lengths[insertion] == Length(10, "mm")
print("bisect insertion for 1 cm:", insertion)

sorted millimeters: [Fraction(1, 1), Fraction(5, 1), Fraction(10, 1), Fraction(20, 1)]
bisect insertion for 1 cm: 2


### Step 6 — An exact-law regression test

Sample-based tests cannot *prove* the contract for every possible rational input. They can, however, catch accidental disagreement between equality, ordering and hashing at important unit-conversion boundaries. We check the trichotomy of three numeric possibilities and equality/hash consistency on a finite representative set.

In [51]:
from itertools import product

samples = [Length("0.1", "cm"), Length(1, "mm"), Length(2, "mm"),
           Length("0.002", "m"), Length(0, "cm")]

for left, right in product(samples, repeat=2):
    assert sum((left < right, left == right, left > right)) == 1
    assert (left <= right) == (left < right or left == right)
    assert (left >= right) == (left > right or left == right)
    if left == right:
        assert hash(left) == hash(right)

for a, b, c in product(samples, repeat=3):
    if a < b and b < c:
        assert a < c

print("capstone: 25 pair tests and 125 triple tests passed")

capstone: 25 pair tests and 125 triple tests passed


### Capstone solution explained

- **Canonical representation:** Exact `Fraction` values in millimeters eliminate unit-dependent equality and float-coercion errors.
- **Equality and hashing:** Both depend solely on `_mm`. Distinct input spellings or units can denote the same length.
- **Ordering:** Exact rational comparison is a genuine total order, making `@total_ordering` suitable.
- **Unsupported operands:** `NotImplemented` permits appropriate dispatch; incompatible raw values eventually produce `TypeError` for ordering.
- **Immutability:** A frozen value cannot change its sorting/hash key after entering a collection.
- **Verification:** We test behavior under direct operators, sorting, binary search, set deduplication and representative algebraic laws.

**Further challenge:** Add inches using the exact scale `Fraction(127, 5)` millimeters per inch; then test that 1 inch equals 25.4 millimeters exactly. Decide how your constructor should validate a `unit` argument of an unhashable type before using it as a dictionary key.

---
# Final review — predict before running

Here are ten conceptual checks. Try answering in a notebook markdown cell or on paper before expanding the explanations below.

1. Why should an equality method return `NotImplemented` instead of `False` for an unfamiliar type?
2. Can two separate instances compare equal when `is` is false?
3. Why can `a > b` work with `__lt__` alone, while `a <= b` fails?
4. Which additional condition must hold when two hashable objects are equal?
5. Why can the right operand's subclass equality method run before the left operand's method?
6. Why is Pareto dominance not a safe input to `@total_ordering`?
7. What breaks when the ordering relation is cyclic?
8. What must remain true of a list before calling `bisect`?
9. Why does componentwise equality need a separate API for an ordinary scalar class?
10. Why are exact unit conversion and input validation part of a comparison design?

## Review answers, with reasoning

1. It declines a comparison and lets the other operand decide; `False` prematurely settles the operation.
2. Yes. `==` asks about the class's equality semantics; `is` checks identity.
3. Python can reflect `>` as `<` with swapped operands. `<=` reflects to `>=`, not to a synthesized expression combining `<` and `==`.
4. Equal hashable values must have equal hashes, including when equality is intentionally supported across types.
5. A proper subclass's overriding reflected comparison can be prioritized so it can implement specialized semantics.
6. Generated total-order operators can treat incomparable values as comparable; the mathematical relation is only partial.
7. Transitivity fails, so no linear ordering can satisfy every comparison edge and sort results lack a meaningful global guarantee.
8. The collection must already be ordered under the same comparison or key policy used by the search.
9. A nonempty tuple of component results can be truthy even if it contains `False`; an ordinary scalar equality should answer one yes/no question.
10. Conversion determines what the values *mean*, while validation keeps unsupported, ambiguous or non-finite values out of the intended ordering domain.

## Final integration check

This final cell checks that definitions from the various tutorial steps still coexist and that the complete notebook executed in the intended order. Successful output is **ALL NEW TUTORIAL CHECKS PASSED**.

In [52]:
assert Badge(7) == LegacyBadge(7)
assert Account("x") != Account("y")
assert SignedDocument("body", "sig") != Document("body")
assert DirectoryKey("Straße") == DirectoryKey("STRASSE")
assert Rational(3, 2) > Rational(1, 2)
assert Instant(utc_time) == Instant(plus_two)
assert not (cheap_slow <= expensive_fast)
assert rock < paper and paper < scissors and not rock < scissors
assert PathLabel(("v", 2)) < PathLabel(("v", 10))
assert bisect.bisect_right(records, probe) == 4
assert min(items).rank == 3 and max(items).rank == 8
assert sorted_by_rich_comparison(rebuilt)
assert (v_a == v_b) is False
assert Length("0.1", "cm") == Length(1, "mm")
print("ALL NEW TUTORIAL CHECKS PASSED")

ALL NEW TUTORIAL CHECKS PASSED


## Practical comparison-design checklist

Before shipping a custom comparable type, ask these questions in order:

1. **Meaning:** What precisely makes two values equal? Is ordering actually meaningful, and is it total or partial?
2. **Supported types:** What does the class accept? Does it return `NotImplemented` for unsupported operands?
3. **Coherence:** Are equality, `<`, any other ordering methods and hashing based on compatible canonical data?
4. **Numerics and text:** Are numeric comparisons exact enough? Is Unicode normalization an intentional policy? Are non-finite or ambiguous values validated?
5. **Mutability:** Can a comparison or hash key change after insertion into a set or sorted list?
6. **Usage:** Would a local `key=` policy be more expressive than giving the class a global order?
7. **Tests:** Have you checked reversed operands, subclass dispatch, equal-but-distinct instances, ties, incomparability, and collection behavior?

**End of tutorial.** All explanations, problems, solutions and automated checks are self-contained in this one notebook.